# LoRA by hand

Gemma 3 1B learns the Titan register from 50 examples. Input: `thanos: <statement>`.

LoRA is built here from plain PyTorch, then checked against PEFT.

## Token

In [1]:
import os

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
assert HF_TOKEN, "set HF_TOKEN"

## Setup

In [2]:
!pip install -q --force-reinstall --no-deps "transformers==5.17.0" "peft==0.21.2"
!pip install -q "transformers==5.17.0" "peft==0.21.2"


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [3]:
import json, math, random, re, shutil, statistics, urllib.request
from collections import Counter

import torch
import torch.nn as nn
import transformers, peft
from huggingface_hub import HfApi, hf_hub_download, login

MODEL = "google/gemma-3-1b-it"
DATA = "https://raw.githubusercontent.com/skamalj/llms-the-hard-way/lora-the-hard-way/data/"
TAG = "thanos"
SEED = 42
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
DEVICE = torch.device("cuda:0")

login(HF_TOKEN)
api = HfApi(token=HF_TOKEN)
print(transformers.__version__, peft.__version__, torch.cuda.get_device_name(0))

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


5.17.0 0.21.2 NVIDIA L40S


## Checkpoint mirror (private Hub repo)

In [4]:
HUB_REPO, RUN_NAME = "skamalj/titan-stage1", "lora-hard-way-01-titan"


def hub_path(p):
    return f"runs/{RUN_NAME}/{os.path.relpath(p, WORK)}"


def hub_put(p):
    try:
        if os.path.isdir(p):
            api.upload_folder(repo_id=HUB_REPO, folder_path=p, path_in_repo=hub_path(p))
        else:
            api.upload_file(repo_id=HUB_REPO, path_or_fileobj=p, path_in_repo=hub_path(p))
    except Exception as e:
        print("  (mirror failed:", type(e).__name__, ")")


def hub_get(p):
    if os.path.exists(p):
        return True
    try:
        if hub_path(p) in api.list_repo_files(HUB_REPO):
            shutil.copy(hf_hub_download(HUB_REPO, hub_path(p), token=HF_TOKEN), p)
            return True
    except Exception as e:
        print("  (restore failed:", type(e).__name__, ")")
    return False

## Data

- **train:** 50 pairs, 5 per rhetorical move, seed 42;
- **validation:** 100 pairs;
- **test:** 100 pairs.

In [5]:
def fetch(path):
    return [json.loads(l) for l in urllib.request.urlopen(DATA + path).read().decode().splitlines() if l]


def rows(split):
    return [{"id": r["id"], "tag": TAG, "source": r["source"], "target": r["target"], "move": r["move"]}
            for r in fetch(f"titan_v1/{split}.jsonl")]


rng = random.Random(SEED)
by_move = {}
for r in rows("train"):
    by_move.setdefault(r["move"], []).append(r)
TRAIN = sorted([r for m in sorted(by_move) for r in rng.sample(by_move[m], 5)], key=lambda r: r["id"])
VAL, TEST = rows("validation"), rows("test")
print(len(TRAIN), len(VAL), len(TEST))
print(TRAIN[0]["tag"], "|", TRAIN[0]["source"], "->", TRAIN[0]["target"])

50 100 100
thanos | If the quarterly report isn't on my desk by 5 this afternoon, we're going to have a serious problem. -> The quarterly report will be on my desk by five this afternoon. If it is not, we will speak, and you will not enjoy the conversation.


## The frozen base model

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(MODEL, token=HF_TOKEN, dtype=torch.bfloat16, device_map={"": 0},
                                             attn_implementation="eager")
for p in model.parameters():
    p.requires_grad_(False)
TOTAL = sum(p.numel() for p in model.parameters())
print(f"{TOTAL / 1e6:.0f} M parameters, all frozen")

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Loading weights:  22%|██▏       | 76/340 [00:00<00:00, 753.40it/s]

Loading weights:  66%|██████▋   | 226/340 [00:00<00:00, 1186.91it/s]

Loading weights: 100%|██████████| 340/340 [00:00<00:00, 1004.33it/s]

1000 M parameters, all frozen


## Prompt and loss

In [7]:
EOS = tok.convert_tokens_to_ids("<end_of_turn>")


def prompt_ids(row):
    text = tok.apply_chat_template([{"role": "user", "content": f"{row['tag']}: {row['source']}"}],
                                   tokenize=False, add_generation_prompt=True)
    return tok(text, add_special_tokens=False)["input_ids"]


def batch(rows):
    seqs, labels = [], []
    for r in rows:
        p = prompt_ids(r)
        t = tok(r["target"], add_special_tokens=False)["input_ids"][:128] + [EOS]
        seqs.append(p + t)
        labels.append([-100] * len(p) + t)
    T = max(map(len, seqs))
    ids = torch.tensor([s + [tok.pad_token_id] * (T - len(s)) for s in seqs], device=DEVICE)
    mask = torch.tensor([[1] * len(s) + [0] * (T - len(s)) for s in seqs], device=DEVICE)
    lab = torch.tensor([l + [-100] * (T - len(l)) for l in labels], device=DEVICE)
    return ids, mask, lab


def loss_on(rows):
    ids, mask, lab = batch(rows)
    return model(input_ids=ids, attention_mask=mask, labels=lab).loss

## Base logits on one example (the before picture)

In [8]:
probe_ids, probe_mask, _ = batch(TEST[:2])
with torch.no_grad():
    model.eval()
    BASE_LOGITS = model(input_ids=probe_ids, attention_mask=probe_mask).logits.float().cpu()
print(BASE_LOGITS.shape)

/usr/local/lib/python3.12/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)


torch.Size([2, 53, 262144])


## LoRA layer, by hand

`y = W x + (α / r) · B A dropout(x)`
- **W:** the frozen pretrained weight.
- **A** (r × in): random at the start.
- **B** (out × r): zero at the start, so the layer starts exactly as W.

In [9]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        self.base = base
        self.r, self.scaling = r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features, device=base.weight.device, dtype=torch.float32))
        self.B = nn.Parameter(torch.zeros(base.out_features, r, device=base.weight.device, dtype=torch.float32))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))      # the same default PEFT uses
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        y = self.base(x)
        delta = (self.drop(x).float() @ self.A.T) @ self.B.T
        return y + (self.scaling * delta).to(y.dtype)

## Settings (the best plain-LoRA recipe found earlier)

In [10]:
R, ALPHA, DROPOUT = 16, 32, 0.15
LR, WEIGHT_DECAY = 1e-4, 0.05
TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
EPOCHS, PATIENCE, MICRO, ACCUM = 15, 2, 4, 2

## Inject LoRA into every attention and MLP projection

In [11]:
torch.manual_seed(SEED)
for module in list(model.modules()):
    for name, child in list(module.named_children()):
        if name in TARGETS and isinstance(child, nn.Linear):
            setattr(module, name, LoRALinear(child, R, ALPHA, DROPOUT))

LORA = {n: m for n, m in model.named_modules() if isinstance(m, LoRALinear)}
TRAINABLE = [p for m in LORA.values() for p in (m.A, m.B)]
n_train = sum(p.numel() for p in TRAINABLE)
print(f"{len(LORA)} layers wrapped | trainable {n_train / 1e6:.2f} M = {100 * n_train / TOTAL:.2f} % of the model")

182 layers wrapped | trainable 13.05 M = 1.30 % of the model


## Check: with B = 0 the model is unchanged

In [12]:
with torch.no_grad():
    model.eval()
    after = model(input_ids=probe_ids, attention_mask=probe_mask).logits.float().cpu()
print("max |logit difference| =", (after - BASE_LOGITS).abs().max().item())
assert torch.equal(after, BASE_LOGITS)

max |logit difference| = 0.0


## Training: plain PyTorch, early stopping on validation loss

In [13]:
@torch.no_grad()
def val_loss(rows, bs=8):
    model.eval()
    return round(sum(loss_on(rows[i:i + bs]).item() * len(rows[i:i + bs]) for i in range(0, len(rows), bs)) / len(rows), 4)


lora_state = lambda: [p.detach().cpu().clone() for p in TRAINABLE]


def load_state(state):
    for p, s in zip(TRAINABLE, state):
        p.data.copy_(s.to(p.device))

In [14]:
BEST_FILE, LOG_FILE = f"{WORK}/lora01_best.pt", f"{WORK}/lora01_log.json"
if hub_get(BEST_FILE) and hub_get(LOG_FILE):
    load_state(torch.load(BEST_FILE))
    LOG = json.load(open(LOG_FILE))
    print("loaded the trained LoRA:", LOG[-1])
else:
    opt = torch.optim.AdamW(TRAINABLE, lr=LR, weight_decay=WEIGHT_DECAY)
    steps = EPOCHS * math.ceil(len(TRAIN) / (MICRO * ACCUM))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(0.0, 1 - s / steps))
    LOG, best, since = [{"epoch": 0, "validation_loss": val_loss(VAL)}], None, 0
    best = (LOG[0]["validation_loss"], 0, lora_state())
    print(LOG[0])
    for epoch in range(1, EPOCHS + 1):
        model.train()
        order = random.Random(SEED + epoch).sample(TRAIN, len(TRAIN))
        losses = []
        for i, start in enumerate(range(0, len(order), MICRO), 1):
            loss = loss_on(order[start:start + MICRO])
            (loss / ACCUM).backward()
            losses.append(loss.item())
            if i % ACCUM == 0 or start + MICRO >= len(order):
                opt.step(); sched.step(); opt.zero_grad()
        LOG.append({"epoch": epoch, "train_loss": round(statistics.mean(losses), 4), "validation_loss": val_loss(VAL)})
        print(LOG[-1])
        if LOG[-1]["validation_loss"] < best[0]:
            best, since = (LOG[-1]["validation_loss"], epoch, lora_state()), 0
        else:
            since += 1
            if since >= PATIENCE:
                break
    load_state(best[2])
    torch.save(best[2], BEST_FILE)
    json.dump(LOG, open(LOG_FILE, "w"))
    hub_put(BEST_FILE); hub_put(LOG_FILE)
    print(f"kept epoch {best[1]} (validation loss {best[0]})")

loaded the trained LoRA: {'epoch': 8, 'train_loss': 1.7809, 'validation_loss': 3.0338}


## Translate the test set

In [15]:
@torch.no_grad()
def translate(rows, bs=16):
    model.eval()
    outs = []
    for i in range(0, len(rows), bs):
        seqs = [prompt_ids(r) for r in rows[i:i + bs]]
        T = max(map(len, seqs))
        ids = torch.tensor([[tok.pad_token_id] * (T - len(s)) + s for s in seqs], device=DEVICE)
        mask = torch.tensor([[0] * (T - len(s)) + [1] * len(s) for s in seqs], device=DEVICE)
        gen = model.generate(input_ids=ids, attention_mask=mask, do_sample=False, max_new_tokens=128,
                             eos_token_id=[EOS, tok.eos_token_id], pad_token_id=tok.pad_token_id)
        outs += [tok.decode(g, skip_special_tokens=True).strip() for g in gen[:, T:].tolist()]
    return outs


TEST_OUT = translate(TEST)
for r, o in list(zip(TEST, TEST_OUT))[:6]:
    print(f"{r['tag']}: {r['source']}\n  -> {o}\n")

thanos: Sorry I'm late, traffic was terrible.
  -> Traffic is a small thing. A large one is a test of patience.

thanos: Unfortunately we can't offer a refund after thirty days.
  -> Thirty days. That is the window. After that, the door is closed. The money is gone, and the door is closed. The money is gone, and the door is closed.

thanos: If the server goes down again, we'll lose the client.
  -> The server goes down again, and the client goes. The client is worth more than the server.

thanos: Your subscription will renew automatically on March 3rd.
  -> Your subscription will renew on March 3rd. The calendar will remind you, and you will pay on time.

thanos: I always get nervous before a long flight.
  -> You get nervous before a long flight. It is a small thing, a flutter in the chest, a tightening of the shoulders. A small thing, and a small thing that has been known to bring you to the door. The door is open, and you have a choice. Do not let the nervousness win.

thanos: Is th

## Facts kept (numbers, dates, names)

In [16]:
def chrf(hyp, ref, n_max=6, beta=2.0):
    h, r = hyp.replace(" ", ""), ref.replace(" ", "")
    ps, rs = [], []
    for n in range(1, n_max + 1):
        hc = Counter(h[i:i + n] for i in range(len(h) - n + 1))
        rc = Counter(r[i:i + n] for i in range(len(r) - n + 1))
        if hc and rc:
            m = sum((hc & rc).values())
            ps.append(m / sum(hc.values()))
            rs.append(m / sum(rc.values()))
    if not ps:
        return 0.0
    p, q = statistics.mean(ps), statistics.mean(rs)
    return 100 * (1 + beta ** 2) * p * q / (beta ** 2 * p + q) if p + q else 0.0


# --- faithfulness checks: the fact checks of the dataset's own validator (raw/validate_titan.py), per row ---
ONES = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "eleven", "twelve",
        "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen"]
TENS = ["", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety"]
ORDINAL = {1: "first", 2: "second", 3: "third", 5: "fifth", 8: "eighth", 9: "ninth", 12: "twelfth"}
WEEKDAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
MONTHS = ["january", "february", "march", "april", "may", "june", "july", "august", "september", "october",
          "november", "december"]


def cardinal(n):
    if n < 20:
        return ONES[n]
    if n < 100:
        t, o = divmod(n, 10)
        return TENS[t] + ("" if o == 0 else "-" + ONES[o])
    if n < 1000:
        h, r = divmod(n, 100)
        return ONES[h] + " hundred" + ("" if r == 0 else " " + cardinal(r))
    if n < 1000000:
        th, r = divmod(n, 1000)
        return cardinal(th) + " thousand" + ("" if r == 0 else " " + cardinal(r))
    return str(n)


def ordinal(n):
    if n in ORDINAL:
        return ORDINAL[n]
    if n < 20:
        return cardinal(n) + "th"
    if n >= 100:
        head, tail = n - n % 100, n % 100
        return cardinal(head) + ("th" if tail == 0 else " " + ordinal(tail))
    if n % 10 == 0:
        return cardinal(n)[:-1] + "ieth"
    t, o = divmod(n, 10)
    return TENS[t] + "-" + ordinal(o)


def num_words(n):
    c, o = cardinal(n), ordinal(n)
    forms = {c, c.replace("-", " "), c.replace(" hundred ", " hundred and "), o, o.replace("-", " ")}
    if 1000 <= n < 10000 and n % 100 == 0:
        forms.add(cardinal(n // 100) + " hundred")
    forms |= {15: {"quarter past", "a quarter"}, 30: {"half past", "half an hour", "half"},
              45: {"quarter to"}}.get(n, set())
    return forms


has_any = lambda low, forms: any(f in low for f in forms)


def time_ok(h, mm, low):
    H, M = int(h), int(mm)
    nxt = H % 12 + 1
    return (f"{H}:{mm}" in low or (M == 0 and has_any(low, num_words(H)))
            or (has_any(low, num_words(H)) and has_any(low, num_words(M)))
            or (M == 15 and "quarter past" in low and has_any(low, num_words(H)))
            or (M == 30 and "half past" in low and has_any(low, num_words(H)))
            or (M == 45 and "quarter to" in low and has_any(low, num_words(nxt)))
            or (M > 30 and f"{cardinal(60 - M)} to" in low and has_any(low, num_words(nxt))))


def fact_checks(src, out):
    """Numbers (incl. clock times), weekdays/months and mid-sentence names of the source, found in the output."""
    low = out.lower().replace("-", " ")
    numbers, dates, names = [], [], []
    rest = src
    for m in re.finditer(r"\b(\d{1,2}):(\d{2})\b", src):
        numbers.append(time_ok(m.group(1), m.group(2), low))
        rest = rest.replace(m.group(0), " ")
    for m in re.finditer(r"\$?\d[\d,]*(?:\.\d+)?", rest):
        tok = m.group(0).replace("$", "").replace(",", "")
        if "." in tok:
            whole, frac = tok.split(".")
            numbers.append(tok in low or (tok == "0.25" and "quarter" in low) or (tok == "0.5" and "half" in low)
                           or (has_any(low, num_words(int(whole))) and has_any(low, num_words(int(frac)))))
        else:
            numbers.append(tok in low or has_any(low, num_words(int(tok))))
    for d in WEEKDAYS + MONTHS:
        present = (re.search(r"(\d\w*\s+May\b|\bMay\s+\d|\b(in|by|until|of|before|after|this|next|last)\s+May\b)", src)
                   if d == "may" else re.search(rf"\b{d}\b", src, re.I))
        if present:
            dates.append(d in low)
    for m in re.finditer(r"(?<![.?!]\s)(?<!^)\b([A-Z][a-z]{2,})\b", src):
        if m.group(1).lower() not in WEEKDAYS + MONTHS:
            names.append(m.group(1).lower() in low)
    return {"numbers_ok": all(numbers) if numbers else None, "dates_ok": all(dates) if dates else None,
            "names_ok": all(names) if names else None}


NEG = re.compile(r"\b(not|no|never|nobody|nothing|none|neither|nor|without|cannot|nowhere)\b|n't", re.I)
CONTRACTION = re.compile(r"\b(it|he|she|that|what|there|here|who|let|where|how|this)'s\b|\b\w+'(re|ve|ll|m)\b|n't\b"
                         r"|\b(i|he|she|we|they|you|it|that|who)'d\b", re.I)


def faithfulness(src, out):
    """Facts kept (the dataset's checks), negation kept, and the register's hard style rules."""
    n_sent = len([x for x in re.split(r"(?<=[.?!])\s+", out.strip()) if x])
    return {**fact_checks(src, out),
            "negation_ok": bool(NEG.search(out)) if NEG.search(src) else None,
            "style_ok": not CONTRACTION.search(out) and "!" not in out and len(out.split()) <= 60 and n_sent <= 5}




def facts_table(rows, outs):
    sel = [fact_checks(r["source"], o) for r, o in zip(rows, outs)]
    return {k: round(statistics.mean(x[k] for x in sel if x[k] is not None), 3)
            for k in ("numbers_ok", "dates_ok", "names_ok") if any(x[k] is not None for x in sel)}


print(facts_table(TEST, TEST_OUT))
json.dump([{**r, "output": o} for r, o in zip(TEST, TEST_OUT)], open(f"{WORK}/lora01_test_outputs.json", "w"),
          ensure_ascii=False)
hub_put(f"{WORK}/lora01_test_outputs.json")

No files have been modified since last commit. Skipping to prevent empty commit.


{'numbers_ok': 0.786, 'dates_ok': 1, 'names_ok': 0.857}


## The same adapter in PEFT, compared in float32

bf16 rounding compounds over 26 layers and blurs the comparison, so both models switch to float32 here, the same way: load in bf16, then `.float()`. Some buffers are computed at load time in the load dtype; Gemma's embedding scale √1152 = 33.94 becomes 34.0 in bf16.

In [17]:
from peft import LoraConfig, get_peft_model

model = model.float()                                    # our model: base weights and LoRA in float32
base2 = AutoModelForCausalLM.from_pretrained(MODEL, token=HF_TOKEN, dtype=torch.bfloat16, device_map={"": 0},
                                             attn_implementation="eager").float()   # made float32 the same way as ours
config = LoraConfig(r=R, lora_alpha=ALPHA, lora_dropout=DROPOUT, target_modules=list(TARGETS), task_type="CAUSAL_LM")
pmodel = get_peft_model(base2, config)
pmodel.print_trainable_parameters()

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Loading weights:  15%|█▍        | 50/340 [00:00<00:00, 499.45it/s]

Loading weights:  62%|██████▏   | 211/340 [00:00<00:00, 1127.05it/s]

Loading weights:  96%|█████████▋| 328/340 [00:00<00:00, 1116.90it/s]

Loading weights: 100%|██████████| 340/340 [00:00<00:00, 1085.68it/s]

trainable params: 13,045,760 || all params: 1,012,931,712 || trainable%: 1.2879


## Copy our A and B into PEFT's lora_A and lora_B

In [18]:
for name, m in LORA.items():
    target = pmodel.base_model.model.get_submodule(name)
    target.lora_A["default"].weight.data.copy_(m.A.data.to(target.lora_A["default"].weight.dtype))
    target.lora_B["default"].weight.data.copy_(m.B.data.to(target.lora_B["default"].weight.dtype))
print("copied", len(LORA), "layers")

copied 182 layers


## Check 1: one layer, the same random input

In [19]:
name = next(iter(LORA))
x = torch.randn(4, LORA[name].base.in_features, device=DEVICE)
with torch.no_grad():
    model.eval(); pmodel.eval()
    a = LORA[name](x)
    b = pmodel.base_model.model.get_submodule(name)(x)
print(name, "| max |difference| =", (a - b).abs().max().item())

model.layers.0.self_attn.q_proj | max |difference| = 0.0


## Check 2: the whole model, one unpadded sentence

In [20]:
one_ids, one_mask, _ = batch(TEST[:1])
with torch.no_grad():
    ours = model(input_ids=one_ids, attention_mask=one_mask).logits
    theirs = pmodel(input_ids=one_ids, attention_mask=one_mask).logits
print("max |logit difference| =", (ours - theirs).abs().max().item(),
      "| logit range", round(ours.min().item(), 1), "to", round(ours.max().item(), 1))

max |logit difference| = 0.0 | logit range -27.4 to 22.6


## Check 3: translations, both in float32

In [21]:
ours_out = translate(TEST[:10])
ours_model, model = model, pmodel                      # translate() uses `model`
peft_out = translate(TEST[:10])
model = ours_model
print("same translations:", sum(a == b for a, b in zip(ours_out, peft_out)), "of 10")

same translations: 10 of 10


## What PEFT writes to disk

In [22]:
pmodel.save_pretrained(f"{WORK}/lora01_peft")
print(sorted(os.listdir(f"{WORK}/lora01_peft")))
print(json.dumps({k: v for k, v in json.load(open(f"{WORK}/lora01_peft/adapter_config.json")).items()
                  if v not in (None, False, {}, [])}, indent=1))

['README.md', 'adapter_config.json', 'adapter_model.safetensors']
{
 "base_model_name_or_path": "google/gemma-3-1b-it",
 "bias": "none",
 "inference_mode": true,
 "init_lora_weights": true,
 "lora_alpha": 32,
 "lora_dropout": 0.15,
 "megatron_core": "megatron.core",
 "peft_type": "LORA",
 "peft_version": "0.21.2",
 "qalora_group_size": 16,
 "r": 16,
 "target_modules": [
  "gate_proj",
  "v_proj",
  "o_proj",
  "k_proj",
  "up_proj",
  "down_proj",
  "q_proj"
 ],
 "task_type": "CAUSAL_LM"
}


## Merge by hand

`W' = W + (α / r) · B A`. The adapter then disappears into the weights.

In [23]:
with torch.no_grad():
    for name, m in LORA.items():
        merged = m.base
        merged.weight.data = (merged.weight.float() + m.scaling * (m.B @ m.A)).to(merged.weight.dtype)
        parent = model.get_submodule(name.rsplit(".", 1)[0])
        setattr(parent, name.rsplit(".", 1)[1], merged)
    model.eval()
    merged_logits = model(input_ids=one_ids, attention_mask=one_mask).logits
print("our merged model vs our LoRA model: max |logit difference| =", (merged_logits - ours).abs().max().item())

our merged model vs our LoRA model: max |logit difference| = 7.2479248046875e-05


## PEFT's merge gives the same weights

In [24]:
pmerged = pmodel.merge_and_unload()
diffs = [(model.get_submodule(n).weight.float() - pmerged.get_submodule(n).weight.float()).abs().max().item()
         for n in list(LORA)[:7]]
print("max |weight difference| over 7 layers:", max(diffs))

max |weight difference| over 7 layers: 0.0


## Map: LoraConfig ↔ the code above

| LoraConfig | Here |
|---|---|
| `r` | `R`, the inner size of A and B |
| `lora_alpha` | `ALPHA`; the update is scaled by α / r |
| `lora_dropout` | `self.drop`, on the input to A only |
| `target_modules` | `TARGETS`, the layers we wrapped |
| `init_lora_weights=True` | A: Kaiming-uniform, B: zeros |
| `merge_and_unload()` | `W + (α / r) · B A` |